# Refactored Comparative Study of Pretrained CNN Models for Plant Disease Classification

This notebook is restructured for midterm presentation clarity.

**Models:** MobileNetV2, ResNet50, VGG16, EfficientNetB0, DenseNet121

Core logic, hyperparameters, and model architectures are unchanged.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU detected: {gpus[0].name}")
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2, ResNet50, VGG16, EfficientNetB0, DenseNet121
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
    matthews_corrcoef,
)
warnings.filterwarnings('ignore')

## 2. Configuration

In [ ]:
# Paths for dataset (unchanged)
DRIVE_DATASET = "/content/drive/MyDrive/Plant_leave_diseases_dataset_with_augmentation"
LOCAL_DATASET = "/content/dataset"

# USE_LOCAL_COPY = True  → copy to /content/dataset (fast training; NEW runtime = one long copy).
# USE_LOCAL_COPY = False → read from Drive only (no copy wait; each epoch slower).
USE_LOCAL_COPY = True

if USE_LOCAL_COPY:
    if not os.path.exists(LOCAL_DATASET):
        print("Copying dataset from Drive to local disk...")
        !rm -rf /content/dataset
        !cp -r "/content/drive/MyDrive/Plant_leave_diseases_dataset_with_augmentation" /content/dataset
        print("Dataset copy completed")
    else:
        print("Dataset already exists on local disk, skipping copy.")
    DATASET_DIR = LOCAL_DATASET
else:
    DATASET_DIR = DRIVE_DATASET
    print("Reading dataset from Drive (no local copy). Slower per epoch, but no copy step.")

IMG_SIZE = (224, 224)
BATCH_SIZE = 24
EPOCHS = 10
LEARNING_RATE = 1e-4

MODEL_SAVE_DIR = "/content/drive/MyDrive/models"
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

print(f"Dataset: {DATASET_DIR}")
print(f"Image size: {IMG_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")

## 3. Common Data Pipeline

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
num_classes = len(class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

print(f"Dataset loaded. Number of classes: {num_classes}")
print(class_names)

## 3.1 Sample images from the dataset

Preview a batch so the notebook is not empty before training and you can show class diversity during evaluation.

In [ ]:
plt.figure(figsize=(16, 10))
for images, labels in train_ds.take(1):
    n_show = min(15, len(images))
    for i in range(n_show):
        ax = plt.subplot(3, 5, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i])], fontsize=7)
        plt.axis("off")
plt.suptitle("Sample training images (one batch)", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Reusable Functions

In [ ]:
def build_model(model_name, num_classes, input_shape=(224, 224, 3)):
    backbone_config = {
        "MobileNetV2": {
            "model_fn": MobileNetV2,
            "preprocess": tf.keras.applications.mobilenet_v2.preprocess_input,
        },
        "ResNet50": {
            "model_fn": ResNet50,
            "preprocess": tf.keras.applications.resnet50.preprocess_input,
        },
        "VGG16": {
            "model_fn": VGG16,
            "preprocess": tf.keras.applications.vgg16.preprocess_input,
        },
        "EfficientNetB0": {
            "model_fn": EfficientNetB0,
            "preprocess": tf.keras.applications.efficientnet.preprocess_input,
        },
        "DenseNet121": {
            "model_fn": DenseNet121,
            "preprocess": tf.keras.applications.densenet.preprocess_input,
        },
    }

    config = backbone_config[model_name]
    base_model = config["model_fn"](
        weights="imagenet",
        include_top=False,
        input_shape=input_shape,
    )
    base_model.trainable = False

    inputs = layers.Input(shape=input_shape)
    x = config["preprocess"](inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs, outputs, name=model_name)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def train_model(model, train_ds, val_ds, model_name):
    print(f"Training {model_name}...")
    start_time = time.time()
    history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)
    elapsed = time.time() - start_time
    print("Training completed")
    print(f"{model_name} training time: {elapsed/60:.2f} minutes")
    return history, elapsed


def evaluate_model(model, val_dataset):
    y_true = []
    y_pred = []

    for images, labels in val_dataset:
        preds = model.predict(images, verbose=0)
        y_true.extend(labels.numpy())
        y_pred.extend(np.argmax(preds, axis=1))

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    accuracy = np.mean(y_true == y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    balanced_acc = balanced_accuracy_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)

    return {
        "y_true": y_true,
        "y_pred": y_pred,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "balanced_accuracy": balanced_acc,
        "mcc": mcc,
        "confusion_matrix": cm,
    }


def plot_training_history(history, model_name):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history.history['accuracy'], label='Train', marker='o')
    axes[0].plot(history.history['val_accuracy'], label='Validation', marker='s')
    axes[0].set_title(f'{model_name} - Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(history.history['loss'], label='Train', marker='o')
    axes[1].plot(history.history['val_loss'], label='Validation', marker='s')
    axes[1].set_title(f'{model_name} - Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


def generate_confusion_matrix(cm, class_names, model_name):
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    plt.figure(figsize=(12, 10))
    sns.heatmap(
        cm_normalized,
        annot=False,
        fmt=".2f",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
    )
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


def generate_classification_report(y_true, y_pred, class_names, model_name):
    print(f"\n{'='*60}")
    print(f"Classification Report - {model_name}")
    print(f"{'='*60}")
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0, digits=4))


def append_comparison_row(model_name, model, results, training_time_sec):
    """Append one row to comparison_rows with extended metrics for the final table."""
    comparison_rows.append({
        "Model": model_name,
        "Val Accuracy": results["accuracy"],
        "Balanced Acc": results["balanced_accuracy"],
        "Precision (w)": results["precision"],
        "Recall (w)": results["recall"],
        "F1 (w)": results["f1_score"],
        "Precision (macro)": results["precision_macro"],
        "Recall (macro)": results["recall_macro"],
        "F1 (macro)": results["f1_macro"],
        "MCC": results["mcc"],
        "Training Time (min)": training_time_sec / 60.0,
        "Training Time (s)": training_time_sec,
        "Parameters": model.count_params(),
    })


histories = {}
trained_models = {}
training_times = {}
all_results = {}
comparison_rows = []

## 5. MobileNetV2

In [ ]:
model_name = "MobileNetV2"
print(f"\n===== {model_name} =====")
mobilenet_model = build_model(model_name, num_classes)
history_mobilenet, time_mobilenet = train_model(mobilenet_model, train_ds, val_ds, model_name)
mobilenet_model.save(f"{MODEL_SAVE_DIR}/{model_name}.h5")

results = evaluate_model(mobilenet_model, val_ds)
plot_training_history(history_mobilenet, model_name)
generate_confusion_matrix(results['confusion_matrix'], class_names, model_name)
generate_classification_report(results['y_true'], results['y_pred'], class_names, model_name)

histories[model_name] = history_mobilenet
trained_models[model_name] = mobilenet_model
training_times[model_name] = time_mobilenet
all_results[model_name] = results
append_comparison_row(model_name, mobilenet_model, results, time_mobilenet)

print(f"{model_name} -> Val Acc: {results['accuracy']:.4f} | Balanced: {results['balanced_accuracy']:.4f} | F1(w): {results['f1_score']:.4f}")
print(f"Saved: {MODEL_SAVE_DIR}/{model_name}.h5")

## 6. ResNet50

In [ ]:
model_name = "ResNet50"
print(f"\n===== {model_name} =====")
resnet_model = build_model(model_name, num_classes)
history_resnet, time_resnet = train_model(resnet_model, train_ds, val_ds, model_name)
resnet_model.save(f"{MODEL_SAVE_DIR}/{model_name}.h5")

results = evaluate_model(resnet_model, val_ds)
plot_training_history(history_resnet, model_name)
generate_confusion_matrix(results['confusion_matrix'], class_names, model_name)
generate_classification_report(results['y_true'], results['y_pred'], class_names, model_name)

histories[model_name] = history_resnet
trained_models[model_name] = resnet_model
training_times[model_name] = time_resnet
all_results[model_name] = results
append_comparison_row(model_name, resnet_model, results, time_resnet)

print(f"{model_name} -> Val Acc: {results['accuracy']:.4f} | Balanced: {results['balanced_accuracy']:.4f} | F1(w): {results['f1_score']:.4f}")
print(f"Saved: {MODEL_SAVE_DIR}/{model_name}.h5")

## 7. VGG16

In [ ]:
model_name = "VGG16"
print(f"\n===== {model_name} =====")
vgg_model = build_model(model_name, num_classes)
history_vgg, time_vgg = train_model(vgg_model, train_ds, val_ds, model_name)
vgg_model.save(f"{MODEL_SAVE_DIR}/{model_name}.h5")

results = evaluate_model(vgg_model, val_ds)
plot_training_history(history_vgg, model_name)
generate_confusion_matrix(results['confusion_matrix'], class_names, model_name)
generate_classification_report(results['y_true'], results['y_pred'], class_names, model_name)

histories[model_name] = history_vgg
trained_models[model_name] = vgg_model
training_times[model_name] = time_vgg
all_results[model_name] = results
append_comparison_row(model_name, vgg_model, results, time_vgg)

print(f"{model_name} -> Val Acc: {results['accuracy']:.4f} | Balanced: {results['balanced_accuracy']:.4f} | F1(w): {results['f1_score']:.4f}")
print(f"Saved: {MODEL_SAVE_DIR}/{model_name}.h5")

## 8. EfficientNetB0

In [ ]:
model_name = "EfficientNetB0"
print(f"\n===== {model_name} =====")
efficientnet_model = build_model(model_name, num_classes)
history_efficientnet, time_efficientnet = train_model(efficientnet_model, train_ds, val_ds, model_name)
efficientnet_model.save(f"{MODEL_SAVE_DIR}/{model_name}.h5")

results = evaluate_model(efficientnet_model, val_ds)
plot_training_history(history_efficientnet, model_name)
generate_confusion_matrix(results['confusion_matrix'], class_names, model_name)
generate_classification_report(results['y_true'], results['y_pred'], class_names, model_name)

histories[model_name] = history_efficientnet
trained_models[model_name] = efficientnet_model
training_times[model_name] = time_efficientnet
all_results[model_name] = results
append_comparison_row(model_name, efficientnet_model, results, time_efficientnet)

print(f"{model_name} -> Val Acc: {results['accuracy']:.4f} | Balanced: {results['balanced_accuracy']:.4f} | F1(w): {results['f1_score']:.4f}")
print(f"Saved: {MODEL_SAVE_DIR}/{model_name}.h5")

## 9. DenseNet121

In [ ]:
model_name = "DenseNet121"
print(f"\n===== {model_name} =====")
densenet_model = build_model(model_name, num_classes)
history_densenet, time_densenet = train_model(densenet_model, train_ds, val_ds, model_name)
densenet_model.save(f"{MODEL_SAVE_DIR}/{model_name}.h5")

results = evaluate_model(densenet_model, val_ds)
plot_training_history(history_densenet, model_name)
generate_confusion_matrix(results['confusion_matrix'], class_names, model_name)
generate_classification_report(results['y_true'], results['y_pred'], class_names, model_name)

histories[model_name] = history_densenet
trained_models[model_name] = densenet_model
training_times[model_name] = time_densenet
all_results[model_name] = results
append_comparison_row(model_name, densenet_model, results, time_densenet)

print(f"{model_name} -> Val Acc: {results['accuracy']:.4f} | Balanced: {results['balanced_accuracy']:.4f} | F1(w): {results['f1_score']:.4f}")
print(f"Saved: {MODEL_SAVE_DIR}/{model_name}.h5")

## 10. Final Comparison (Same as Previous Style)

In [ ]:
comparison_df = pd.DataFrame(comparison_rows)
comparison_df = comparison_df.sort_values(by="Val Accuracy", ascending=False).reset_index(drop=True)

print("\n" + "=" * 90)
print("COMPARATIVE STUDY - MODEL PERFORMANCE SUMMARY")
print("=" * 90)
print(comparison_df.to_string(index=False))
print("=" * 90)

comparison_df.to_csv(f"{MODEL_SAVE_DIR}/comparison_table.csv", index=False)
print(f"Saved comparison table: {MODEL_SAVE_DIR}/comparison_table.csv")

In [ ]:
plt.figure(figsize=(12, 6))
for model_name, history in histories.items():
    plt.plot(history.history['val_accuracy'], label=model_name, marker='o')

plt.title('Validation Accuracy Comparison Across Models')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{MODEL_SAVE_DIR}/val_accuracy_all_models.png", dpi=150, bbox_inches='tight')
plt.show()

plt.figure(figsize=(12, 6))
for model_name, history in histories.items():
    plt.plot(history.history['val_loss'], label=model_name, marker='o')

plt.title('Validation Loss Comparison Across Models')
plt.xlabel('Epoch')
plt.ylabel('Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{MODEL_SAVE_DIR}/val_loss_all_models.png", dpi=150, bbox_inches='tight')
plt.show()

model_names = list(comparison_df['Model'])
accuracies = list(comparison_df['Val Accuracy'])
f1_w = list(comparison_df['F1 (w)'])
f1_macro = list(comparison_df['F1 (macro)'])
bal_acc = list(comparison_df['Balanced Acc'])

x = np.arange(len(model_names))
width = 0.22

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x - width, accuracies, width, label='Val accuracy', color='steelblue')
ax.bar(x, f1_w, width, label='F1 (weighted)', color='coral')
ax.bar(x + width, f1_macro, width, label='F1 (macro)', color='seagreen')
ax.bar(x + 2 * width, bal_acc, width, label='Balanced acc', color='mediumpurple')

ax.set_title('Model comparison — accuracy & F1 variants')
ax.set_ylabel('Score')
ax.set_xticks(x + width / 2)
ax.set_xticklabels(model_names, rotation=15, ha='right')
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{MODEL_SAVE_DIR}/bar_metrics_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

# Training time (minutes) — horizontal bar
times_min = list(comparison_df['Training Time (min)'])
fig, ax = plt.subplots(figsize=(10, 5))
order = np.argsort(times_min)
ax.barh([model_names[i] for i in order], [times_min[i] for i in order], color='teal', edgecolor='black', linewidth=0.3)
ax.set_xlabel('Total training time (minutes)')
ax.set_title('Training time per model (full 5-epoch run)')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{MODEL_SAVE_DIR}/training_time_barh.png", dpi=150, bbox_inches='tight')
plt.show()

# Training time — vertical bar (same data, presentation alternative)
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(model_names, times_min, color='darkcyan', edgecolor='black', linewidth=0.3)
ax.set_ylabel('Minutes')
ax.set_title('Training time per model (vertical)')
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(f"{MODEL_SAVE_DIR}/training_time_barv.png", dpi=150, bbox_inches='tight')
plt.show()

### Extra comparison visuals

Heatmap of core metrics, Matthews correlation coefficient (MCC) per model, radar overlay, and accuracy vs training time.

In [ ]:
# Heatmap: metrics in [0, 1] range (good for side-by-side comparison)
metric_cols_01 = [
    "Val Accuracy",
    "Balanced Acc",
    "F1 (w)",
    "F1 (macro)",
    "Precision (w)",
    "Recall (w)",
    "Precision (macro)",
    "Recall (macro)",
]
hm_data = comparison_df.set_index("Model")[metric_cols_01].T

plt.figure(figsize=(12, 6))
sns.heatmap(hm_data, annot=True, fmt=".3f", cmap="YlGnBu", vmin=0, vmax=1)
plt.title("Metric heatmap — all models (0–1 scale)")
plt.ylabel("Metric")
plt.tight_layout()
plt.savefig(f"{MODEL_SAVE_DIR}/metrics_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

# MCC (can be negative; shown separately)
plt.figure(figsize=(10, 4))
sns.barplot(x="Model", y="MCC", data=comparison_df, palette="viridis", edgecolor="black")
plt.axhline(0, color="gray", linewidth=0.8)
plt.title("Matthews correlation coefficient (MCC) — higher is better")
plt.xticks(rotation=20, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{MODEL_SAVE_DIR}/mcc_by_model.png", dpi=150, bbox_inches="tight")
plt.show()

# Radar chart: four main scores per model
labels_r = ["Val Acc", "Balanced", "F1 (w)", "F1 (macro)"]
n_ax = len(labels_r)
angles = np.linspace(0, 2 * np.pi, n_ax, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for _, row in comparison_df.iterrows():
    vals = [
        row["Val Accuracy"],
        row["Balanced Acc"],
        row["F1 (w)"],
        row["F1 (macro)"],
    ]
    vals += vals[:1]
    ax.plot(angles, vals, "o-", linewidth=1.5, label=row["Model"])
    ax.fill(angles, vals, alpha=0.06)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels_r)
ax.set_ylim(0, 1)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.05))
plt.title("Radar — key metrics (0–1)")
plt.tight_layout()
plt.savefig(f"{MODEL_SAVE_DIR}/radar_metrics.png", dpi=150, bbox_inches="tight")
plt.show()

# Trade-off: accuracy vs training cost
plt.figure(figsize=(9, 6))
plt.scatter(
    comparison_df["Training Time (min)"],
    comparison_df["Val Accuracy"],
    s=140,
    c=range(len(comparison_df)),
    cmap="tab10",
    edgecolors="black",
)
for _, row in comparison_df.iterrows():
    plt.annotate(
        row["Model"],
        (row["Training Time (min)"], row["Val Accuracy"]),
        textcoords="offset points",
        xytext=(6, 4),
        fontsize=9,
    )
plt.xlabel("Training time (minutes)")
plt.ylabel("Validation accuracy")
plt.title("Accuracy vs training time")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{MODEL_SAVE_DIR}/accuracy_vs_time.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. XAI — Grad-CAM heatmap (MobileNetV2)

**Explainable AI:** Grad-CAM highlights image regions that most influenced the model’s prediction. Uses your saved `MobileNetV2.h5` weights (no retraining).

In [ ]:
import cv2

def _get_mobilenet_backbone(model):
    # The MobileNetV2 backbone is nested as a sub-model layer inside `build_model()`.
    for layer in model.layers:
        if isinstance(layer, tf.keras.Model) and "mobilenetv2" in layer.name.lower():
            return layer
    raise ValueError("Could not find MobileNetV2 backbone sub-model in the model layers.")


def find_last_conv_layer(model):
    backbone = _get_mobilenet_backbone(model)
    conv_types = (tf.keras.layers.Conv2D, tf.keras.layers.DepthwiseConv2D)

    # Scan backbone layers from end to start to get the last conv-like layer
    for layer in reversed(backbone.layers):
        if isinstance(layer, conv_types):
            return layer.name

    raise ValueError("No convolutional layer found in MobileNetV2 backbone.")

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """Robust Grad-CAM for your trained MobileNetV2 classifier.

    Fix: compute gradients in a *connected* graph by:
    1) extracting target conv features
    2) running the remaining backbone
    3) running the classifier head

    This avoids Keras `KeyError` issues when using multi-output models.
    """
    backbone = _get_mobilenet_backbone(model)
    target_layer = backbone.get_layer(last_conv_layer_name)

    preprocess_fn = tf.keras.applications.mobilenet_v2.preprocess_input
    x = preprocess_fn(img_array)

    # Extractors
    target_extractor = tf.keras.Model(
        inputs=backbone.input,
        outputs=target_layer.output,
    )
    tail_extractor = tf.keras.Model(
        inputs=target_layer.output,
        outputs=backbone.output,
    )
    head_extractor = tf.keras.Model(
        inputs=backbone.output,
        outputs=model.output,
    )

    with tf.GradientTape() as tape:
        conv_output = target_extractor(x, training=False)
        backbone_features = tail_extractor(conv_output, training=False)
        preds = head_extractor(backbone_features, training=False)

        if pred_index is None:
            pred_index = tf.argmax(preds[0])

        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_output = conv_output[0]  # (H, W, C)
    heatmap = tf.reduce_sum(conv_output * pooled_grads, axis=-1)

    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

# Load trained MobileNetV2 model WITHOUT deserializing custom layers
# (fixes errors like: Unknown layer: 'TrueDivide')
mobile_model_path = f"{MODEL_SAVE_DIR}/MobileNetV2.h5"

mobile_model = build_model("MobileNetV2", num_classes)
mobile_model.load_weights(mobile_model_path)

# Select last convolutional layer automatically
last_conv_layer_name = find_last_conv_layer(mobile_model)
print("Last conv layer:", last_conv_layer_name)

# Load one sample image from validation dataset
for batch_images, batch_labels in val_ds.take(1):
    sample_image = batch_images[0].numpy().astype("uint8")
    sample_label = int(batch_labels[0].numpy())
    break

# Prepare image for model input
img_array = tf.expand_dims(tf.cast(sample_image, tf.float32), axis=0)

# Predict class using your trained model
preds = mobile_model(img_array, training=False)
pred_index = int(tf.argmax(preds[0]).numpy())
true_index = sample_label
print(f"Predicted class: {class_names[pred_index]}")
print(f"True class: {class_names[true_index]}")

# Generate heatmap for the predicted class
heatmap = make_gradcam_heatmap(img_array, mobile_model, last_conv_layer_name, pred_index=pred_index)

# Overlay heatmap on original image
heatmap_resized = cv2.resize(heatmap, (sample_image.shape[1], sample_image.shape[0]))
heatmap_uint8 = np.uint8(255 * heatmap_resized)
heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
overlay = cv2.addWeighted(cv2.cvtColor(sample_image, cv2.COLOR_RGB2BGR), 0.6, heatmap_color, 0.4, 0)
overlay = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)

# Display
plt.figure(figsize=(14, 4))
plt.subplot(1, 3, 1)
plt.imshow(sample_image)
plt.title(f"Original\nTrue: {class_names[true_index]}")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(heatmap_resized, cmap="jet")
plt.title("XAI — Grad-CAM heatmap")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(overlay)
plt.title("Overlay (image + heatmap)")
plt.axis("off")

plt.suptitle("Explainable AI (XAI): class-discriminative localization", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(f"{MODEL_SAVE_DIR}/xai_gradcam_mobilenetv2.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. XAI — Grad-CAM heatmap (MobileNetV2)

**Explainable AI:** Grad-CAM highlights image regions that most influenced the model’s prediction. Uses your saved `MobileNetV2.h5` weights (no retraining).

## 13. Alternative explainability (no Grad-CAM)

If Grad-CAM fails on nested models, these still work for your report:

1. **Top-k class probabilities** — which diseases the model considers most likely.  
2. **Occlusion sensitivity** — graying out image patches; large drops mean that region mattered.  
3. **Misclassification gallery** — concrete failure cases (true vs predicted label).

In [ ]:
# Load MobileNetV2 (same weights as training) — safe even if you skipped the Grad-CAM cell
xai_model = build_model("MobileNetV2", num_classes)
xai_model.load_weights(f"{MODEL_SAVE_DIR}/MobileNetV2.h5")

# --- 1) Top-k probabilities on a grid of validation images ---
k_top = 3
n_show = 9

fig, axes = plt.subplots(3, 3, figsize=(14, 12))
axes = axes.flatten()
batch_imgs, batch_labs = next(iter(val_ds.take(1)))
probs = xai_model.predict(batch_imgs, verbose=0)

for idx in range(min(n_show, len(batch_imgs))):
    ax = axes[idx]
    im = batch_imgs[idx].numpy().astype("uint8")
    ax.imshow(im)
    true_i = int(batch_labs[idx].numpy())
    p = probs[idx]
    top_idx = np.argsort(p)[::-1][:k_top]
    title_lines = [f"True: {class_names[true_i][:30]}..."] if len(class_names[true_i]) > 30 else [f"True: {class_names[true_i]}"]
    for rank, ci in enumerate(top_idx, 1):
        title_lines.append(f"{rank}. {class_names[ci][:28]} ({p[ci]:.2f})")
    ax.set_title("\n".join(title_lines), fontsize=7)
    ax.axis("off")
plt.suptitle("Top-3 predicted classes (MobileNetV2)", fontsize=12)
plt.tight_layout()
plt.savefig(f"{MODEL_SAVE_DIR}/xai_topk_probs.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- 2) Occlusion sensitivity (one image): darker = occluding that patch hurt confidence more ---
import cv2

for batch_imgs, batch_labs in val_ds.take(1):
    occ_img = batch_imgs[0].numpy().astype(np.float32)
    true_lab = int(batch_labs[0].numpy())
    break

h, w = IMG_SIZE[0], IMG_SIZE[1]
patch = 40
stride = 20

base_in = np.expand_dims(occ_img, 0)
base_prob = xai_model.predict(base_in, verbose=0)[0]
target_cls = int(np.argmax(base_prob))  # explain model's decision

rows = (h - patch) // stride + 1
cols = (w - patch) // stride + 1
sensitivity = np.zeros((rows, cols))

for ri, i in enumerate(range(0, h - patch + 1, stride)):
    for cj, j in enumerate(range(0, w - patch + 1, stride)):
        occluded = occ_img.copy()
        occluded[i : i + patch, j : j + patch] = 127.5
        p = xai_model.predict(np.expand_dims(occluded, 0), verbose=0)[0]
        sensitivity[ri, cj] = base_prob[target_cls] - p[target_cls]

sens_up = cv2.resize(sensitivity, (w, h), interpolation=cv2.INTER_CUBIC)
sens_norm = (sens_up - sens_up.min()) / (sens_up.max() - sens_up.min() + 1e-8)

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(occ_img.astype("uint8"))
plt.title(f"Original\nTrue: {class_names[true_lab]}")
plt.axis("off")
plt.subplot(1, 3, 2)
plt.imshow(sens_norm, cmap="magma")
plt.title("Occlusion map\n(brighter = more important for prediction)")
plt.axis("off")
plt.subplot(1, 3, 3)
plt.imshow(occ_img.astype("uint8"))
plt.imshow(sens_norm, cmap="jet", alpha=0.45)
plt.title(f"Overlay\nPred focus: {class_names[target_cls]}")
plt.axis("off")
plt.tight_layout()
plt.savefig(f"{MODEL_SAVE_DIR}/xai_occlusion.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- 3) Misclassification gallery (first N wrong predictions from validation batches) ---
max_wrong = 12
max_batches = 30
wrong_imgs = []
wrong_true = []
wrong_pred = []

for batch_imgs, batch_labs in val_ds.take(max_batches):
    preds = xai_model.predict(batch_imgs, verbose=0)
    pred_idx = np.argmax(preds, axis=1)
    labs = batch_labs.numpy()
    for bi in range(len(labs)):
        if pred_idx[bi] != labs[bi]:
            wrong_imgs.append(batch_imgs[bi].numpy().astype("uint8"))
            wrong_true.append(int(labs[bi]))
            wrong_pred.append(int(pred_idx[bi]))
        if len(wrong_imgs) >= max_wrong:
            break
    if len(wrong_imgs) >= max_wrong:
        break

n = len(wrong_imgs)
if n == 0:
    print("No misclassifications in sampled batches (model doing very well on this slice).")
else:
    cols = min(4, n)
    rows = (n + cols - 1) // cols
    plt.figure(figsize=(4 * cols, 3.5 * rows))
    for i in range(n):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(wrong_imgs[i])
        plt.title(f"T: {class_names[wrong_true[i]][:20]}\nP: {class_names[wrong_pred[i]][:20]}", fontsize=7)
        plt.axis("off")
    plt.suptitle("Misclassified examples (sample from validation)", fontsize=12)
    plt.tight_layout()
    plt.savefig(f"{MODEL_SAVE_DIR}/xai_misclassified_gallery.png", dpi=150, bbox_inches="tight")
    plt.show()